### Libraries and Depencies

In [ ]:
from typing import List, Dict, Any
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import logging
import time
import re
import os
import spacy




HF_TOKEN = os.getenv('HF_TOKEN') or globals().get('HF_TOKEN')  # Hugging Face API token from environment variable
print(f"Hugging Face API Token: {'Set' if HF_TOKEN else 'Not Set'}")


# Optional packages for assignment requirements
# pip install pinecone-client rank-bm25
try:
    from pinecone import Pinecone, ServerlessSpec
except Exception:
    Pinecone = None
    ServerlessSpec = None

try:
    from rank_bm25 import BM25Okapi
except Exception:
    BM25Okapi = None

### Chunking Strategy

In [ ]:
# Arbitary Size Chunking
def Arbitary_chunking(text, chunk_size=512, overlap=64):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks

In [ ]:
# Recursive Chunking
def Recursive_chunking(text, target_size=800):
    sections = text.split('\n## ')
    chunks = []
    for section in sections:
        if len(section) > 1000:
            paragraphs = section.split('\n\n')
            current_chunk = ""
            for para in paragraphs:
                if len(current_chunk + para) < target_size:
                    current_chunk += para + "\n\n"
                else:
                    if current_chunk.strip():
                        chunks.append(current_chunk.strip())
                    current_chunk = para + "\n\n"
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
        else:
            if section.strip():
                chunks.append(section.strip())
    return chunks

In [ ]:
# Semantic Chunking
def Semantic_chunking(text, max_chars=800):
    try:
        import spacy
        nlp = spacy.load('en_core_web_sm')
        doc = nlp(text)
        sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
    except Exception:
        # Fallback if spaCy model is unavailable
        sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

    chunks = []
    current_chunk = ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) < max_chars:
            current_chunk += sentence + " "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

def chunk_document(text, method='recursive'):
    if method == 'arbitary':
        return Arbitary_chunking(text)
    if method == 'semantic':
        return Semantic_chunking(text)
    return Recursive_chunking(text)

### Document Retrival

In [ ]:
class HybridRetriever:
    """
    Hybrid retrieval with BM25 + Semantic (Pinecone) using RRF + optional re-ranking.
    """

    def __init__(self, index_name='rag-assignment3-index', embedding_model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'):
        self.embedding_model = SentenceTransformer(embedding_model_name)
        self.index_name = index_name
        self.pc_client = None
        self.pc_index = None

        self.bm25_model = None
        self.bm25_chunks = []

        self.intent_labels = ['factual query', 'procedural steps', 'conceptual explanation']
        self.classifier = pipeline(
            'zero-shot-classification',
            model='cross-encoder/nli-deberta-v3-small'
        )

        pinecone_api_key = os.getenv('PINECONE_API_KEY')
        pinecone_env = os.getenv('PINECONE_ENVIRONMENT', 'us-east-1')

        if Pinecone is not None and pinecone_api_key:
            self.pc_client = Pinecone(api_key=pinecone_api_key)
            existing_indexes = [idx.name for idx in self.pc_client.list_indexes()]
            if index_name not in existing_indexes:
                self.pc_client.create_index(
                    name=index_name,
                    dimension=384,
                    metric='cosine',
                    spec=ServerlessSpec(cloud='aws', region=pinecone_env)
                )
            self.pc_index = self.pc_client.Index(index_name)

    def classify_query(self, query: str) -> str:
        prediction = self.classifier(query, self.intent_labels)
        top_label = prediction['labels'][0].lower()
        if 'factual' in top_label:
            return 'factual'
        if 'procedural' in top_label:
            return 'procedural'
        return 'conceptual'

    def set_bm25_corpus(self, corpus_chunks: list):
        """Build BM25 corpus using normalized tokens from chunk text."""
        self.corpus = corpus_chunks
        if BM25Okapi is None:
            self.bm25 = None
            return

        tokenized_corpus = [
            [word for word in re.split(r'\W+', chunk.get('text', '').lower()) if word]
            for chunk in self.corpus
        ]
        self.bm25 = BM25Okapi(tokenized_corpus) if tokenized_corpus else None

    def _bm25_search(self, query: str, top_k: int) -> list:
        """BM25 retrieval using the same regex tokenization strategy as indexing."""
        if not getattr(self, 'bm25', None):
            return []

        tokenized_query = [word for word in re.split(r'\W+', query.lower()) if word]
        scores = self.bm25.get_scores(tokenized_query)

        top_indices = np.argsort(scores)[::-1][:top_k]
        return [
            {
                'id': self.corpus[i]['id'],
                'text': self.corpus[i]['text'],
                'source': self.corpus[i].get('source', 'unknown'),
                'score': float(scores[i]),
                'search_type': 'keyword'
            }
            for i in top_indices
        ]

    def _semantic_search(self, query: str, top_k: int) -> List[Dict[str, Any]]:
        if self.pc_index is None:
            return []
        query_vector = self.embedding_model.encode(query).tolist()
        response = self.pc_index.query(
            vector=query_vector,
            top_k=top_k,
            include_metadata=True
        )

        results = []
        for m in response.matches:
            meta = m.metadata or {}
            results.append({
                'id': m.id,
                'text': meta.get('text', ''),
                'source': meta.get('source', 'unknown'),
                'score': float(m.score),
                'search_type': 'semantic'
            })
        return results

    def _rrf_fusion(self, keyword_results, semantic_results, k=60):
        rrf_scores = {}
        merged_docs = {}

        for rank, doc in enumerate(keyword_results, start=1):
            doc_id = doc['id']
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank)
            merged_docs[doc_id] = doc

        for rank, doc in enumerate(semantic_results, start=1):
            doc_id = doc['id']
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank)
            if doc_id in merged_docs:
                merged_docs[doc_id]['search_type'] = 'hybrid'
            else:
                merged_docs[doc_id] = doc

        fused = []
        for doc_id, score in rrf_scores.items():
            doc = dict(merged_docs[doc_id])
            doc['rrf_score'] = score
            fused.append(doc)

        fused.sort(key=lambda x: x.get('rrf_score', 0.0), reverse=True)
        return fused

    def _rerank(self, query: str, results: list, top_k: int) -> list:
        """Optimized to use BATCH vectorization for massive speedup."""
        if not results:
            return []

        # 1. Encode the query
        query_emb = self.embedding_model.encode(query)

        # 2. Extract ALL texts and encode them in a SINGLE batch (Huge speedup)
        texts_to_encode = [doc.get('text', '') for doc in results]
        doc_embeddings = self.embedding_model.encode(texts_to_encode)

        # 3. Calculate cosine similarity for the whole batch at once
        similarities = cosine_similarity([query_emb], doc_embeddings)[0]

        # 4. Re-assign scores and sort
        for idx, doc in enumerate(results):
            doc['rerank_score'] = similarities[idx]

        results.sort(key=lambda x: x['rerank_score'], reverse=True)
        return results[:top_k]

    def retrieve_hybrid(self, query: str, top_k: int = 5, rerank: bool = True) -> List[Dict[str, Any]]:
        query_type = self.classify_query(query)
        if query_type == 'factual':
            keyword_results = self._bm25_search(query, top_k=8)
            semantic_results = self._semantic_search(query, top_k=4)
        else:
            keyword_results = self._bm25_search(query, top_k=4)
            semantic_results = self._semantic_search(query, top_k=8)

        fused = self._rrf_fusion(keyword_results, semantic_results)
        if rerank:
            return self._rerank(query, fused, top_k=top_k)
        return fused[:top_k]

    def retrieve_semantic_only(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        return self._semantic_search(query, top_k=top_k)

In [ ]:
utility_embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

def get_embedding(text):
    return utility_embedding_model.encode(text)

def count_tokens(text):
    # Simple approximation for notebook experimentation
    return len(text.split())

def remove_duplicates(retrieved_chunks: list) -> list:
    """Removes duplicate dictionaries from a list based on their 'id'."""
    seen_ids = set()
    unique_chunks = []
    
    for chunk in retrieved_chunks:
        if chunk['id'] not in seen_ids:
            seen_ids.add(chunk['id'])
            unique_chunks.append(chunk)
            
    return unique_chunks

def adds_new_info(new_chunk_emb: np.ndarray, selected_embs: list, threshold: float = 0.85) -> bool:
    """
    Checks if a chunk adds new info by comparing pre-calculated embeddings.
    """
    if not selected_embs:
        return True
        
    # Check similarity against all already selected chunks
    for selected_emb in selected_embs:
        sim = cosine_similarity([new_chunk_emb], [selected_emb])[0][0]
        if sim > threshold:
            return False # Too similar to an existing chunk
            
    return True

In [ ]:
def intelligent_context_selection(retrieved_chunks: list, embedding_model, max_tokens: int = 1500) -> list:
    """
    Select diverse chunks under a token budget and return chunk dictionaries.
    """
    selected_chunks = []
    selected_embs = []  # Cache to store embeddings we've already calculated
    current_length = 0

    for chunk in retrieved_chunks:
        text = chunk.get('text', '')
        if not text:
            continue

        chunk_length = len(text.split())
        if current_length + chunk_length > max_tokens:
            continue

        # Get embedding for the current chunk just ONCE
        new_emb = embedding_model.encode(text)

        # Pass the pre-calculated embeddings to our helper
        if adds_new_info(new_emb, selected_embs):
            selected_chunks.append(chunk)
            selected_embs.append(new_emb)  # Store the embedding so we don't recalculate it later
            current_length += chunk_length

    return selected_chunks

In [ ]:
def create_rag_prompt(query, context_chunks):
    context_text = "\n\n".join([
        f"Source {i+1}: {chunk['text']}"
        for i, chunk in enumerate(context_chunks)
    ])

    prompt = f"""CONTEXT:
{context_text}

QUESTION: {query}

INSTRUCTIONS:
1. Answer only from the provided context.
2. If information is missing, say clearly what is missing.
3. Keep answer concise and factual.
"""
    return prompt

def generate_answer_hf(prompt, hf_model='Qwen/Qwen2.5-7B-Instruct'):
    from huggingface_hub import InferenceClient

    hf_token = os.getenv('HF_TOKEN') or globals().get('HF_TOKEN')
    candidate_models = [hf_model, 'google/flan-t5-large', 'bigscience/bloom-560m']
    last_error = None

    if hf_token and hf_token != 'your_hf_token_here':
        client = InferenceClient(provider='hf-inference', api_key=hf_token)
        for model_name in candidate_models:
            start = time.time()
            try:
                out = client.text_generation(
                    prompt,
                    model=model_name,
                    max_new_tokens=350,
                    temperature=0.2
                )
                latency = time.time() - start
                return out, latency
            except Exception as e:
                last_error = repr(e)
                continue

    # Local fallback when HF Inference is unavailable.
    global _local_gen_pipe
    if '_local_gen_pipe' not in globals() or _local_gen_pipe is None:
        _local_gen_pipe = pipeline('text-generation', model='distilgpt2')

    # Truncate prompt to fit local model context window
    local_prompt = prompt
    try:
        tok = _local_gen_pipe.tokenizer
        max_positions = int(getattr(_local_gen_pipe.model.config, 'n_positions', 1024))
        max_input_tokens = max(64, max_positions - 120)
        token_ids = tok.encode(prompt, add_special_tokens=False)
        if len(token_ids) > max_input_tokens:
            token_ids = token_ids[-max_input_tokens:]
            local_prompt = tok.decode(token_ids, skip_special_tokens=True)
    except Exception:
        pass

    start = time.time()
    local_out = _local_gen_pipe(local_prompt, max_new_tokens=120, do_sample=False)
    latency = time.time() - start
    if isinstance(local_out, list) and len(local_out) > 0:
        if 'generated_text' in local_out[0]:
            return local_out[0]['generated_text'], latency
        return str(local_out[0]), latency

    if last_error:
        return f'Fallback output unavailable. Remote error: {last_error}', latency
    return str(local_out), latency

### Logging

In [ ]:
class RAGLogger:
    def __init__(self):
        self.logger = logging.getLogger('rag_system')
        self.logger.setLevel(logging.INFO)
        if not self.logger.handlers:
            handler = logging.StreamHandler()
            handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
            self.logger.addHandler(handler)

    def log_rag_interaction(self, query, retrieved_chunks, response, metrics=None, user_feedback=None):
        log_entry = {
            'timestamp': datetime.now().isoformat(),
            'query': query,
            'retrieved_chunks': [
                {
                    'id': chunk.get('id', 'na'),
                    'score': chunk.get('score', chunk.get('rerank_score', 0.0)),
                    'source': chunk.get('source', 'unknown')
                } for chunk in retrieved_chunks
            ],
            'response': response,
            'metrics': metrics or {},
            'user_feedback': user_feedback,
            'retrieval_count': len(retrieved_chunks)
        }
        self.logger.info(f'RAG_INTERACTION: {log_entry}')

    def analyze_failures(self):
        # Extend later by querying persisted logs
        pass

### Corpus Loading and Indexing

In [ ]:
# TODO: Add your corpus folder path here later
CORPUS_PATH = r'C:\\Users\\Dell\\Desktop\\Semester 6\\NLP with Deep Learning\\Retrieval Augmented Generation\\synthetic_knowledge_items.csv'

CHUNKING_METHOD = 'recursive'   # 'arbitary' | 'recursive' | 'semantic'
TOP_K = 5

def read_corpus_documents(corpus_path):
    corpus_path = Path(corpus_path)
    docs = []

    # Helper to load a plain text/markdown file
    def _load_text_file(fp):
        text = fp.read_text(encoding='utf-8', errors='ignore').strip()
        if text:
            docs.append({'source': str(fp), 'text': text})

    # Helper to load a CSV file into one document per row
    def _load_csv_file(fp):
        df = pd.read_csv(fp)
        text_cols = [c for c in df.columns if df[c].dtype == 'object']
        if len(text_cols) == 0:
            text_cols = list(df.columns)

        for idx, row in df.iterrows():
            row_parts = []
            for col in text_cols:
                val = row.get(col, None)
                if pd.notna(val):
                    s = str(val).strip()
                    if s:
                        row_parts.append(f'{col}: {s}')
            if row_parts:
                docs.append({
                    'source': f'{fp}#row={idx}',
                    'text': '\n'.join(row_parts)
                })

    if corpus_path.is_file():
        suffix = corpus_path.suffix.lower()
        if suffix in ['.txt', '.md']:
            _load_text_file(corpus_path)
        elif suffix == '.csv':
            _load_csv_file(corpus_path)
        return docs

    # If a directory is provided, scan supported file types recursively
    for pattern in ['*.txt', '*.md']:
        for fp in corpus_path.rglob(pattern):
            _load_text_file(fp)

    for fp in corpus_path.rglob('*.csv'):
        _load_csv_file(fp)

    return docs

In [ ]:
def build_chunks_from_docs(docs, method='recursive'):
    chunks = []
    chunk_counter = 0
    for doc in docs:
        local_chunks = chunk_document(doc['text'], method=method)
        for ch in local_chunks:
            chunks.append({
                'id': f'ch_{chunk_counter}',
                'text': ch,
                'source': doc['source']
            })
            chunk_counter += 1
    return chunks

def upsert_chunks_to_pinecone(retriever: HybridRetriever, chunks, batch_size=100):
    if retriever.pc_index is None:
        print('Pinecone not configured. Set PINECONE_API_KEY and retry.')
        return

    vectors = []
    for chunk in chunks:
        vec = retriever.embedding_model.encode(chunk['text']).tolist()
        vectors.append({
            'id': chunk['id'],
            'values': vec,
            'metadata': {'text': chunk['text'], 'source': chunk['source']}
        })

    for i in range(0, len(vectors), batch_size):
        retriever.pc_index.upsert(vectors=vectors[i:i + batch_size])

def prepare_retrieval_indexes(corpus_path, chunking_method='recursive'):
    docs = read_corpus_documents(corpus_path)
    if len(docs) == 0:
        raise ValueError('No documents found. Add files in your corpus path and rerun.')

    chunks = build_chunks_from_docs(docs, method=chunking_method)
    retriever = HybridRetriever()
    retriever.set_bm25_corpus(chunks)
    upsert_chunks_to_pinecone(retriever, chunks)

    print(f'Documents loaded: {len(docs)}')
    print(f'Chunks created: {len(chunks)}')
    return retriever, docs, chunks

### LLM-as-a-Judge Evaluation (Faithfulness + Relevancy)

In [ ]:
def call_hf_judge(prompt, model='Qwen/Qwen2.5-7B-Instruct'):
    from huggingface_hub import InferenceClient

    hf_token = os.getenv('HF_TOKEN') or globals().get('HF_TOKEN')
    candidate_models = [model, 'google/flan-t5-large', 'bigscience/bloom-560m']
    last_error = None

    if hf_token and hf_token != 'your_hf_token_here':
        client = InferenceClient(provider='hf-inference', api_key=hf_token)
        for model_name in candidate_models:
            try:
                out = client.text_generation(
                    prompt,
                    model=model_name,
                    max_new_tokens=256,
                    temperature=0.0
                )
                return out
            except Exception as e:
                last_error = repr(e)
                continue

    # Local fallback when HF Inference is unavailable.
    global _local_judge_pipe
    if '_local_judge_pipe' not in globals() or _local_judge_pipe is None:
        _local_judge_pipe = pipeline('text-generation', model='distilgpt2')

    # Truncate prompt to fit local model context window
    local_prompt = prompt
    try:
        tok = _local_judge_pipe.tokenizer
        max_positions = int(getattr(_local_judge_pipe.model.config, 'n_positions', 1024))
        max_input_tokens = max(64, max_positions - 80)
        token_ids = tok.encode(prompt, add_special_tokens=False)
        if len(token_ids) > max_input_tokens:
            token_ids = token_ids[-max_input_tokens:]
            local_prompt = tok.decode(token_ids, skip_special_tokens=True)
    except Exception:
        pass

    local_out = _local_judge_pipe(local_prompt, max_new_tokens=80, do_sample=False)
    if isinstance(local_out, list) and len(local_out) > 0:
        if 'generated_text' in local_out[0]:
            return local_out[0]['generated_text']
        return str(local_out[0])

    if last_error:
        return f'Fallback output unavailable. Remote error: {last_error}'
    return str(local_out)

def extract_claims(answer_text):
    prompt = f"""Extract atomic factual claims from the answer.
Return only a JSON array of short claims.
Answer: {answer_text}"""
    out = call_hf_judge(prompt)
    try:
        arr_match = re.search(r'\[.*\]', out, re.DOTALL)
        if arr_match:
            return list(pd.json_normalize(eval(arr_match.group(0))).values.flatten())
    except Exception:
        pass
    claims = [line.strip('- ').strip() for line in out.split('\n') if line.strip()]
    return [c for c in claims if len(c) > 5][:8]

def verify_claims_against_context(claims, context_text):
    verdicts = []
    for claim in claims:
        prompt = f"""Context:\n{context_text}\n\nClaim: {claim}\n\nIs this claim supported by context? Reply only with SUPPORTED or UNSUPPORTED."""
        out = call_hf_judge(prompt).upper()
        supported = 'SUPPORTED' in out and 'UNSUPPORTED' not in out
        verdicts.append({'claim': claim, 'supported': supported})
    return verdicts

def faithfulness_score(answer_text, retrieved_chunks):
    context_text = '\n\n'.join([c['text'] for c in retrieved_chunks])
    claims = extract_claims(answer_text)
    if len(claims) == 0:
        return 0.0, []
    verdicts = verify_claims_against_context(claims, context_text)
    score = sum(v['supported'] for v in verdicts) / len(verdicts)
    return score, verdicts

def generate_alternate_queries(answer_text):
    prompt = f"""Generate 3 alternative user questions that would have answer below.
Return only one question per line.\n\nAnswer:\n{answer_text}"""
    out = call_hf_judge(prompt)
    lines = [l.strip(' -').strip() for l in out.split('\n') if l.strip()]
    return lines[:3]

def relevancy_score(original_query, answer_text, embedding_model=None):
    if embedding_model is None:
        embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

    alt_qs = generate_alternate_queries(answer_text)
    if len(alt_qs) == 0:
        return 0.0, []

    q_vec = embedding_model.encode(original_query)
    sims = []
    for q in alt_qs:
        q2_vec = embedding_model.encode(q)
        sims.append(float(cosine_similarity([q_vec], [q2_vec])[0][0]))

    return float(np.mean(sims)), sims

In [ ]:
def run_single_query_pipeline(query, retriever, retrieval_mode='hybrid', top_k=5):
    t0 = time.time()
    if retrieval_mode == 'semantic':
        retrieved = retriever.retrieve_semantic_only(query, top_k=top_k)
    else:
        retrieved = retriever.retrieve_hybrid(query, top_k=top_k, rerank=True)
    retrieval_latency = time.time() - t0

    selected_chunks = intelligent_context_selection(retrieved, retriever.embedding_model, max_tokens=2000)
    prompt = create_rag_prompt(query, selected_chunks)

    answer, generation_latency = generate_answer_hf(prompt)
    faith_score, claim_verification = faithfulness_score(answer, selected_chunks)
    rel_score, rel_sims = relevancy_score(query, answer, retriever.embedding_model)

    total_latency = retrieval_latency + generation_latency
    metrics = {
        'faithfulness': faith_score,
        'relevancy': rel_score,
        'retrieval_latency': retrieval_latency,
        'generation_latency': generation_latency,
        'total_latency': total_latency
    }

    return {
        'query': query,
        'answer': answer,
        'retrieved_chunks': selected_chunks,
        'claim_verification': claim_verification,
        'relevancy_similarities': rel_sims,
        'metrics': metrics
    }

### Ablation Study (Chunking + Retrieval)

In [ ]:
def run_ablation(test_queries, corpus_path):
    settings = [
        # {'chunking': 'arbitary', 'retrieval': 'semantic'}, 
        # {'chunking': 'recursive', 'retrieval': 'semantic'},
        # {'chunking': 'recursive', 'retrieval': 'hybrid'},
        {'chunking': 'semantic', 'retrieval': 'hybrid'}
    ]

    rows = []
    for cfg in settings:
        retriever, _, _ = prepare_retrieval_indexes(corpus_path, chunking_method=cfg['chunking'])

        faithfulness_vals = []
        relevancy_vals = []
        latency_vals = []

        for q in test_queries:
            out = run_single_query_pipeline(
                query=q,
                retriever=retriever,
                retrieval_mode=cfg['retrieval'],
                top_k=TOP_K
            )
            faithfulness_vals.append(out['metrics']['faithfulness'])
            relevancy_vals.append(out['metrics']['relevancy'])
            latency_vals.append(out['metrics']['total_latency'])

        rows.append({
            'chunking': cfg['chunking'],
            'retrieval': cfg['retrieval'],
            'faithfulness_avg': float(np.mean(faithfulness_vals)),
            'relevancy_avg': float(np.mean(relevancy_vals)),
            'latency_avg_sec': float(np.mean(latency_vals))
        })

    return pd.DataFrame(rows).sort_values(['faithfulness_avg', 'relevancy_avg'], ascending=False)

In [ ]:
# Example fixed test queries for assignment evaluation (10-20 recommended)
TEST_QUERIES = [
    # --- Multi-Hop & Comparative (Tests Semantic Understanding) ---
    "What is the difference in the setup process between configuring an iOS device versus an Android device for company email?",
    "If I am setting up a new user, do I create their JIRA account before or after assigning their Oracle security roles?",
    
    # --- Keyword-Heavy / Acronyms (Tests BM25 / Lexical Search) ---
    "What is the exact server address (e.g., mail.company.com) needed for the Exchange configuration?",
    "Where exactly do I find the MDM profile settings in the settings app?",
    
    # --- Conversational / Frustrated User (Tests Intent & Noise Filtering) ---
    "I totally forgot my network password and I'm locked out. I don't have access to my backup email right now, what am I supposed to do?",
    "My Microsoft Office isn't just freezing, it's completely crashing my whole computer every time I open Word. How do I fix it?",
    
    # --- Standard Procedural (Baseline Checks) ---
    "What are the exact steps to configure VPN access for remote workers?",
    "What information is required to create a new IT Incident or problem report?",
    "I am an IT specialist; what details do I need to gather before creating an IT request for a new employee?"
]

# Run these after setting CORPUS_PATH and API keys
retriever, docs, chunks = prepare_retrieval_indexes(CORPUS_PATH, chunking_method=CHUNKING_METHOD)
sample_output = run_single_query_pipeline(TEST_QUERIES[0], retriever, retrieval_mode='hybrid', top_k=TOP_K)

print('\n=== SAMPLE QUERY ===')
print(sample_output['query'])
print('\n=== GENERATED ANSWER ===')
print(sample_output['answer'])
print('\n=== METRICS ===')
print(sample_output['metrics'])
print('\n=== TOP CONTEXT CHUNKS (preview) ===')
for i, c in enumerate(sample_output['retrieved_chunks'][:3], start=1):
    print(f'[{i}]', c.get('text', '')[:250], '...')

ablation_df = run_ablation(TEST_QUERIES, CORPUS_PATH)
ablation_df

### Live Web Interface (Streamlit Export Template)

In [ ]:
APP_TEMPLATE = '''
import os
import streamlit as st

st.set_page_config(page_title='RAG Assignment 3', layout='wide')
st.title('RAG-based Question Answering System')

st.markdown('Set environment variables: HF_TOKEN, PINECONE_API_KEY, PINECONE_ENVIRONMENT')
st.markdown('Use the same retrieval and evaluation functions from your notebook in app.py.')

query = st.text_input('Ask a question:')

if query:
    st.info('Connect this app with your notebook pipeline functions.')
    st.write('Generated Answer: ...')
    st.write('Retrieved Context: ...')
    st.write('Faithfulness Score: ...')
    st.write('Relevancy Score: ...')
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(APP_TEMPLATE)

print('app.py template written. Fill in pipeline calls and deploy to Hugging Face Spaces.')

In [ ]:
# Quick smoke test (single query) before full ablation
retriever, docs, chunks = prepare_retrieval_indexes(CORPUS_PATH, chunking_method=CHUNKING_METHOD)
smoke_output = run_single_query_pipeline(TEST_QUERIES[0], retriever, retrieval_mode='hybrid', top_k=TOP_K)
smoke_output['metrics']

In [ ]:
# HF connectivity sanity check
from huggingface_hub import HfApi
hf_token_check = os.getenv('HF_TOKEN') or globals().get('HF_TOKEN')
print('Token present:', bool(hf_token_check))
if hf_token_check:
    try:
        who = HfApi().whoami(token=hf_token_check)
        print('Authenticated as:', who.get('name', 'unknown'))
    except Exception as e:
        print('HF whoami failed:', repr(e))

In [ ]:
# HF generation probe
from huggingface_hub import InferenceClient
probe_client = InferenceClient(token=(os.getenv('HF_TOKEN') or globals().get('HF_TOKEN')))
for probe_model in ['google/flan-t5-base', 'google/flan-t5-large', 'bigscience/bloom-560m']:
    try:
        probe_out = probe_client.text_generation('Hello', model=probe_model, max_new_tokens=10)
        print(probe_model, 'OK', probe_out)
    except Exception as e:
        print(probe_model, 'FAIL', repr(e))